# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Inspect the input data

First, I inspect the schemas of the citation and patent DataFrames to identify the columns needed for the joins.

In [8]:
citations.printSchema()
patents.printSchema()

root
 |-- CITING: integer (nullable = true)
 |-- CITED: integer (nullable = true)

root
 |-- PATENT: integer (nullable = true)
 |-- GYEAR: integer (nullable = true)
 |-- GDATE: integer (nullable = true)
 |-- APPYEAR: integer (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- POSTATE: string (nullable = true)
 |-- ASSIGNEE: integer (nullable = true)
 |-- ASSCODE: integer (nullable = true)
 |-- CLAIMS: integer (nullable = true)
 |-- NCLASS: integer (nullable = true)
 |-- CAT: integer (nullable = true)
 |-- SUBCAT: integer (nullable = true)
 |-- CMADE: integer (nullable = true)
 |-- CRECEIVE: integer (nullable = true)
 |-- RATIOCIT: double (nullable = true)
 |-- GENERAL: double (nullable = true)
 |-- ORIGINAL: double (nullable = true)
 |-- FWDAPLAG: double (nullable = true)
 |-- BCKGTLAG: double (nullable = true)
 |-- SELFCTUB: double (nullable = true)
 |-- SELFCTLB: double (nullable = true)
 |-- SECDUPBD: double (nullable = true)
 |-- SECDLWBD: double (nullable = true)



## Create a patent-state lookup

I select the patent number and state columns needed for the citation joins.

In [9]:
patent_states = patents.select(
    col("PATENT"),
    col("POSTATE")
)

patent_states.show(5)

+-------+-------+
| PATENT|POSTATE|
+-------+-------+
|3070801|   NULL|
|3070802|     TX|
|3070803|     IL|
|3070804|     OH|
|3070805|     CA|
+-------+-------+
only showing top 5 rows



## Find the state of each cited patent

I join the citation data with the patent-state table using the cited patent number to determine the state of each cited patent.

In [10]:
cited_join = citations.join(
    patent_states,
    citations.CITED == patent_states.PATENT,
    "left"
).select(
    citations.CITING,
    citations.CITED,
    patent_states.POSTATE.alias("CITED_STATE")
)

cited_join.show(10)

+-------+-------+-----------+
| CITING|  CITED|CITED_STATE|
+-------+-------+-----------+
|3858242|1515701|       NULL|
|3858242|3319261|         OH|
|3858241|3634889|         OH|
|3858241| 956203|       NULL|
|3858241|1324234|       NULL|
|3858243|2949611|       NULL|
|3858243|3146465|         MI|
|3858241|3398406|         FL|
|3858241|3557384|         MA|
|3858242|3668705|         WI|
+-------+-------+-----------+
only showing top 10 rows



## Find the state of each citing patent

I join the intermediate citation data with the patent-state table again, this time using the citing patent number. This gives the states of both patents involved in each citation.

In [11]:
cj = cited_join.alias("cj")
ps = patent_states.alias("ps")

citation_states = cj.join(
    ps,
    col("cj.CITING") == col("ps.PATENT"),
    "left"
).select(
    col("cj.CITED"),
    col("cj.CITED_STATE"),
    col("cj.CITING"),
    col("ps.POSTATE").alias("CITING_STATE")
)

citation_states.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|1515701|       NULL|3858242|          MI|
|3319261|         OH|3858242|          MI|
|3668705|         WI|3858242|          MI|
|3707004|         WI|3858242|          MI|
|2949611|       NULL|3858243|        NULL|
|3146465|         MI|3858243|        NULL|
|3634889|         OH|3858241|          MA|
| 956203|       NULL|3858241|          MA|
|1324234|       NULL|3858241|          MA|
|3398406|         FL|3858241|          MA|
+-------+-----------+-------+------------+
only showing top 10 rows



## Identify same-state citations

I keep citations where both patents have state information and their states match.

In [12]:
same_state = citation_states.filter(
    col("CITED_STATE").isNotNull() &
    col("CITING_STATE").isNotNull() &
    (col("CITED_STATE") == col("CITING_STATE"))
)

same_state.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|3764357|         AK|3869295|          AK|
|3648767|         AK|3908753|          AK|
|3217791|         AK|4067198|          AK|
|3706204|         AK|4067198|          AK|
|3464385|         AK|4178878|          AK|
|4014293|         AK|4180012|          AK|
|3863553|         AK|4184416|          AK|
|4048808|         AK|4192630|          AK|
|3860151|         AK|4207996|          AK|
|3725182|         AK|4253905|          AK|
+-------+-----------+-------+------------+
only showing top 10 rows



## Count same-state citations

I group the matching citations by the citing patent and count how many same-state patents each patent cites.

In [13]:
same_state_counts = (
    same_state
    .groupBy("CITING")
    .count()
    .withColumnRenamed("count", "SAME_STATE")
)

same_state_counts.show(10)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|3968160|         1|
|5179246|         1|
|5392941|         3|
|5854697|         1|
|4765387|         1|
|5734569|         2|
|3861180|         1|
|4094479|         1|
|4635660|         2|
|4732042|         1|
+-------+----------+
only showing top 10 rows



## Add same-state counts to the patent data

I join the same-state citation counts back to the original patent DataFrame. Patents without a same-state citation count are assigned zero.

In [14]:
p = patents.alias("p")
sc = same_state_counts.alias("sc")

final_patents = p.join(
    sc,
    col("p.PATENT") == col("sc.CITING"),
    "left"
).select(
    "p.*",
    col("sc.SAME_STATE")
).fillna(
    0,
    subset=["SAME_STATE"]
)

final_patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|         0|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    63| NULL|       9|    NULL| 0.3704|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|         0|
|3070805| 1963| 1096|   NULL| 

## Top 10 patents by same-state citations

Finally, I sort the augmented patent data by SAME_STATE in descending order and display the ten patents with the highest counts.

In [15]:
top10 = final_patents.orderBy(
    col("SAME_STATE").desc()
)

top10.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 